# Picking a Reasoning Model for Document QA

This notebook evaluates candidate reasoning models on a small, concrete task:

> Given a folder of course notes, retrieve relevant passages and answer questions using only those passages.

The notebook is designed for teaching and for cheap iteration. It runs in mock mode by default, using a copied folder of Obsidian notes and simulated provider responses. Live API runners are included later, but they are disabled unless you explicitly turn them on.


## What the provider docs recommend

The common recommendation across current provider docs and cookbooks is simpler than a full RAG framework:

1. Keep the corpus small and local for the eval.
2. Define one notebook-owned tool, `search_notes(query, k)`, that searches the folder and returns compact passages with stable source IDs.
3. Let the model call that tool, execute it in Python, then send the result back for a final answer.
4. Log retrieval hits, answer quality, citations, latency, token usage, and estimated cost.

Provider-specific notes used while building this notebook:

- OpenAI: use the Responses API with custom function calling for tool traces and `reasoning.effort` sweeps. Sources: [Responses migration](https://developers.openai.com/api/docs/guides/migrate-to-responses), [function calling](https://developers.openai.com/api/docs/guides/function-calling), [reasoning models](https://developers.openai.com/api/docs/guides/reasoning), [retrieval](https://developers.openai.com/api/docs/guides/retrieval), [agent evals](https://developers.openai.com/api/docs/guides/agent-evals).
- Anthropic: use the Messages API with client-side tools; tool results should immediately follow tool-use messages. Sources: [tool use overview](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview), [define tools](https://platform.claude.com/docs/en/agents-and-tools/tool-use/define-tools), [handle tool calls](https://platform.claude.com/docs/en/agents-and-tools/tool-use/handle-tool-calls), [writing tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents), [Claude cookbooks RAG](https://github.com/anthropics/claude-cookbooks/tree/main/capabilities/retrieval_augmented_generation).
- Google Gemini: use the official Gen AI SDK and Interactions API for tool steps and `thinking_level`; local function tools are better than hosted File Search for eval transparency. Sources: [function calling](https://ai.google.dev/gemini-api/docs/function-calling), [thinking](https://ai.google.dev/gemini-api/docs/thinking), [file search](https://ai.google.dev/gemini-api/docs/file-search), [Gemini cookbook](https://github.com/google-gemini/cookbook).
- DeepSeek: use the OpenAI-compatible Chat Completions API with function tools; `deepseek-v4-pro` is the normalized model from the draft. Sources: [chat completion](https://api-docs.deepseek.com/api/create-chat-completion), [function calling](https://api-docs.deepseek.com/guides/function_calling), [models and pricing](https://api-docs.deepseek.com/quick_start/pricing).
- Z.ai / GLM: use OpenAI-compatible chat completions or the native SDK; GLM-5.2 supports long context and controllable reasoning effort. Sources: [Z.ai docs index](https://docs.z.ai/llms.txt), [GLM-5 repo](https://github.com/zai-org/GLM-5), [OpenAI-compatible Python guide](https://docs.z.ai/guides/develop/openai/python.md), [function calling](https://docs.z.ai/guides/capabilities/function-calling.md).


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import re
import tempfile
import textwrap
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "oreilly_reasoning_model_eval_matplotlib"),
)

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})


In [ ]:
# Draft candidates from the original notebook, normalized to provider API-style model IDs.
DRAFT_MODEL_NAMES = [
    "gemini 3.1 pro preview",
    "glm 5.2",
    "deep seek v4 pro",
    "claude opu 4.8",
    "gpt-5.5",
]

MODEL_CANDIDATES = [
    {"provider": "OpenAI", "model": "gpt-5.5", "draft_name": "gpt-5.5"},
    {"provider": "Anthropic", "model": "claude-opus-4-8", "draft_name": "claude opu 4.8"},
    {"provider": "Google", "model": "gemini-3.1-pro-preview", "draft_name": "gemini 3.1 pro preview"},
    {"provider": "Z.ai", "model": "glm-5.2", "draft_name": "glm 5.2"},
    {"provider": "DeepSeek", "model": "deepseek-v4-pro", "draft_name": "deep seek v4 pro"},
]

# Keep this False for class demos. Turn it on only after checking keys, prices, and limits.
RUN_LIVE_API_CALLS = False
LIVE_CASE_LIMIT = 2
RETRIEVAL_K = 3
MAX_OUTPUT_TOKENS = 650

pd.DataFrame(MODEL_CANDIDATES)


## Load the notes corpus

The folder below is a copied subset of the Obsidian vault notes. It is intentionally small: 22 markdown files, under the requested 30-note limit.


In [ ]:
def find_docs_dir() -> Path:
    here = Path.cwd()
    candidates = [
        here / "data" / "reasoning_model_docs_v2",
        here / "notebooks" / "data" / "reasoning_model_docs_v2",
        here.parent / "notebooks" / "data" / "reasoning_model_docs_v2",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find notebooks/data/reasoning_model_docs_v2")

DOCS_DIR = find_docs_dir()
manifest_path = DOCS_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

print(f"Docs folder: {DOCS_DIR}")
print(f"Markdown notes: {len([p for p in DOCS_DIR.glob('*.md')])}")
pd.DataFrame(manifest).sort_values("file").head(10)


## Build a tiny local search tool

This is deliberately not a full RAG pipeline. The model gets a single tool named `search_notes`; the notebook owns retrieval and logging.


In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "have",
    "how", "i", "in", "is", "it", "its", "of", "on", "or", "that", "the", "their",
    "this", "to", "was", "what", "when", "where", "which", "who", "why", "with", "you",
    "your", "about", "into", "than", "then", "they", "them", "these", "those", "using",
}


def clean_obsidian_markdown(text: str) -> str:
    text = re.sub(r"!\[\[[^\]]+\]\]", "", text)
    text = re.sub(r"\[\[([^|\]]+)\|([^\]]+)\]\]", r"\2", text)
    text = re.sub(r"\[\[([^\]]+)\]\]", r"\1", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def tokenize(text: str) -> list[str]:
    tokens = re.findall(r"[a-zA-Z][a-zA-Z0-9_\-]+", text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]


def make_chunks(doc_id: str, title: str, text: str, max_chars: int = 1100) -> list[dict[str, Any]]:
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks: list[dict[str, Any]] = []
    current: list[str] = []
    current_len = 0

    for paragraph in paragraphs:
        if current and current_len + len(paragraph) > max_chars:
            chunk_text = "\n\n".join(current)
            chunks.append({
                "doc_id": doc_id,
                "chunk_id": f"{doc_id}#chunk-{len(chunks) + 1}",
                "title": title,
                "text": chunk_text,
                "terms": Counter(tokenize(title + " " + chunk_text)),
            })
            current = []
            current_len = 0
        current.append(paragraph)
        current_len += len(paragraph)

    if current:
        chunk_text = "\n\n".join(current)
        chunks.append({
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}#chunk-{len(chunks) + 1}",
            "title": title,
            "text": chunk_text,
            "terms": Counter(tokenize(title + " " + chunk_text)),
        })
    return chunks


def load_documents(docs_dir: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    documents = []
    chunks = []
    title_by_file = {row["file"]: row["source_title"] for row in manifest}
    for path in sorted(docs_dir.glob("*.md")):
        raw = path.read_text(encoding="utf-8", errors="ignore")
        text = clean_obsidian_markdown(raw)
        title = title_by_file.get(path.name, path.stem.replace("_", " ").title())
        doc = {"doc_id": path.name, "title": title, "text": text, "chars": len(text)}
        documents.append(doc)
        chunks.extend(make_chunks(path.name, title, text))
    return documents, chunks


documents, chunks = load_documents(DOCS_DIR)
print(f"Loaded {len(documents)} docs into {len(chunks)} chunks")
pd.DataFrame(documents).sort_values("chars", ascending=False).head(8)


In [ ]:
def score_chunk(query: str, chunk: dict[str, Any]) -> float:
    q_terms = tokenize(query)
    if not q_terms:
        return 0.0
    c_terms: Counter[str] = chunk["terms"]
    overlap = sum(min(3, c_terms.get(term, 0)) for term in q_terms)
    title_terms = set(tokenize(chunk["title"]))
    title_bonus = 0.75 * sum(1 for term in q_terms if term in title_terms)
    phrase_bonus = 0.0
    lower_text = chunk["text"].lower()
    for phrase in re.findall(r"[a-zA-Z][a-zA-Z0-9\- ]{5,}", query.lower()):
        phrase = phrase.strip()
        if len(phrase.split()) >= 2 and phrase in lower_text:
            phrase_bonus += 1.0
    length_norm = math.sqrt(max(20, sum(c_terms.values())))
    return (overlap + title_bonus + phrase_bonus) / length_norm


def search_notes(query: str, k: int = RETRIEVAL_K) -> list[dict[str, Any]]:
    scored = []
    for chunk in chunks:
        score = score_chunk(query, chunk)
        if score > 0:
            scored.append((score, chunk))
    scored.sort(key=lambda item: item[0], reverse=True)
    hits = []
    for score, chunk in scored[:k]:
        snippet = re.sub(r"\s+", " ", chunk["text"]).strip()
        hits.append({
            "doc_id": chunk["doc_id"],
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "score": round(score, 4),
            "text": snippet[:900],
        })
    return hits


def tool_payload(hits: list[dict[str, Any]]) -> str:
    return json.dumps(hits, ensure_ascii=False, indent=2)

sample_hits = search_notes("How should I evaluate LLM agents beyond final answers?", k=3)
print(tool_payload(sample_hits))


## Create a 10-question eval set

Each case has a question, a short gold answer, expected source files, and keyword groups for lightweight automatic grading.


In [ ]:
TASKS = [
    {
        "id": "q01_eval_start",
        "question": "When starting an LLM task, what is the recommended first step before building a larger evaluation dataset?",
        "expected_answer": "Start ad hoc by testing different inputs, then organize interesting eval examples into a small dataset.",
        "expected_sources": ["start_incrementally_on_your_llm_task.md"],
        "expected_keywords": [["ad hoc", "ad-hoc"], ["different inputs"], ["interesting eval examples", "examples"], ["small dataset"]],
    },
    {
        "id": "q02_gold_miniset",
        "question": "What is a gold mini-set in prompt evaluation, and why keep it around 5 to 20 cases?",
        "expected_answer": "A gold mini-set is a small repeatable test set of 5 to 20 cases used for cheap checks before rollout.",
        "expected_sources": ["evaluate_prompts_like_finance_product.md"],
        "expected_keywords": [["gold"], ["5", "20"], ["cheap", "repeatable"], ["rollout"]],
    },
    {
        "id": "q03_prompt_metrics",
        "question": "Which task metrics should the finance-product prompt evaluation lesson track?",
        "expected_answer": "Track accuracy, completeness, citation coverage, time saved, cost, latency, and error rate.",
        "expected_sources": ["evaluate_prompts_like_finance_product.md"],
        "expected_keywords": [["accuracy"], ["completeness"], ["citation coverage"], ["time saved"], ["cost"], ["latency"], ["error rate"]],
    },
    {
        "id": "q04_judge_bias",
        "question": "Name two ways to reduce bias when using an LLM as a judge.",
        "expected_answer": "Use pairwise comparisons, swap answer order to control position bias, allow ties, and control response length.",
        "expected_sources": ["llm_as_judge_suggestions.md"],
        "expected_keywords": [["pairwise"], ["position bias", "swap"], ["ties"], ["response length", "longer responses"]],
    },
    {
        "id": "q05_tool_responses",
        "question": "What should high-quality tool responses return to an agent?",
        "expected_answer": "They should return high-signal, meaningful context that is token efficient and directly useful for the agent's next action.",
        "expected_sources": ["writing_mcp_tools_for_llm_agents.md", "sample_principles_good_llm_agents_anthropic.md"],
        "expected_keywords": [["meaningful context", "high-signal"], ["token"], ["tool"], ["agent"]],
    },
    {
        "id": "q06_agent_eval_steps",
        "question": "How should LLM agents be evaluated beyond checking the final answer?",
        "expected_answer": "Evaluate intermediate steps: correct action, correct action input, correct sequence of steps, and the efficiency of that sequence.",
        "expected_sources": ["evaluation_of_llm_agents.md"],
        "expected_keywords": [["intermediate"], ["correct action"], ["action input"], ["sequence"], ["efficient", "efficiency"]],
    },
    {
        "id": "q07_thinking_budget",
        "question": "Why should you control a reasoning model's thinking budget?",
        "expected_answer": "Not every prompt needs deep thinking; over-thinking wastes reasoning tokens, money, and context window capacity.",
        "expected_sources": ["controlling_the_thinking_budget.md", "reasoning_token_pricing.md"],
        "expected_keywords": [["not all prompts", "not every prompt"], ["wastes", "over-thinking", "overthinking"], ["tokens"], ["money", "cost"], ["context window"]],
    },
    {
        "id": "q08_reasoning_benchmarks",
        "question": "What two families of reasoning benchmarks are listed in the notes?",
        "expected_answer": "The notes group reasoning benchmarks into coding tasks and math tasks with verifiable answers.",
        "expected_sources": ["reasoning_benchmarks.md"],
        "expected_keywords": [["coding"], ["math"], ["verifiable"]],
    },
    {
        "id": "q09_grpo",
        "question": "What is the core idea behind GRPO?",
        "expected_answer": "Sample a group of completions, score them with verifiable rewards, and judge each completion relative to its group instead of using a learned value critic.",
        "expected_sources": ["grpo.md"],
        "expected_keywords": [["group"], ["completions"], ["reward"], ["relative"], ["critic", "value"]],
    },
    {
        "id": "q10_data_types",
        "question": "What three types of data can LLM applications connect to?",
        "expected_answer": "They can connect to structured, semi-structured, and unstructured data.",
        "expected_sources": ["types_of_data_to_connect_with_llms.md"],
        "expected_keywords": [["structured"], ["semi-structured"], ["unstructured"]],
    },
]

tasks_df = pd.DataFrame(TASKS)
tasks_df[["id", "question", "expected_sources"]]


In [ ]:
def retrieval_hit_at_k(question: str, expected_sources: list[str], k: int = RETRIEVAL_K) -> tuple[int, list[str]]:
    hits = search_notes(question, k=k)
    retrieved = [hit["doc_id"] for hit in hits]
    hit = int(any(source in retrieved for source in expected_sources))
    return hit, retrieved

baseline_rows = []
for task in TASKS:
    hit, retrieved = retrieval_hit_at_k(task["question"], task["expected_sources"], k=RETRIEVAL_K)
    baseline_rows.append({
        "id": task["id"],
        "retrieval_hit@3": hit,
        "expected_sources": task["expected_sources"],
        "retrieved_sources": retrieved,
    })

baseline_df = pd.DataFrame(baseline_rows)
print(f"Baseline retrieval hit@3: {baseline_df['retrieval_hit@3'].mean():.0%}")
baseline_df


## Scoring helpers

This is a deliberately small eval, so the grading is lightweight and transparent. In production, replace `keyword_score` with human review, pairwise LLM-as-judge, or a stronger rubric.


In [ ]:
def stable_random(*parts: str) -> float:
    key = "|".join(parts).encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    return int(digest[:12], 16) / float(16 ** 12)


def keyword_score(answer: str, expected_keywords: list[Any]) -> float:
    answer_l = answer.lower()
    if not expected_keywords:
        return 0.0
    hits = 0
    for group in expected_keywords:
        options = group if isinstance(group, list) else [group]
        if any(str(option).lower() in answer_l for option in options):
            hits += 1
    return hits / len(expected_keywords)


def retrieved_source_hit(retrieved_sources: list[str], expected_sources: list[str]) -> int:
    return int(any(source in set(retrieved_sources) for source in expected_sources))


def citation_score(answer: str, retrieved_sources: list[str]) -> float:
    answer_l = answer.lower()
    mentioned = [source for source in retrieved_sources if source.lower() in answer_l]
    return 1.0 if mentioned else 0.0


def estimate_tokens(text: str) -> int:
    return max(1, math.ceil(len(text) / 4))


## Mock model profiles

These profiles are not benchmark claims. They create deterministic example data for teaching the eval loop and visualizations without spending API credits.

Before using this notebook for a real decision, run the live cells on your own dataset and replace the illustrative pricing with the provider pricing page for the day you run it.


In [ ]:
MODEL_PROFILES = [
    {
        "provider": "OpenAI",
        "model": "gpt-5.5",
        "reasoning_control": "reasoning.effort='low' for default eval; sweep medium/high later",
        "answer_quality": 0.95,
        "retrieval_quality": 0.94,
        "base_latency_s": 2.1,
        "input_usd_per_mtok": 2.50,
        "output_usd_per_mtok": 15.00,
        "color": "#10a37f",
    },
    {
        "provider": "Anthropic",
        "model": "claude-opus-4-8",
        "reasoning_control": "thinking/adaptive effort for hard cases; disabled for cheap eval",
        "answer_quality": 0.96,
        "retrieval_quality": 0.91,
        "base_latency_s": 2.6,
        "input_usd_per_mtok": 5.00,
        "output_usd_per_mtok": 25.00,
        "color": "#d97757",
    },
    {
        "provider": "Google",
        "model": "gemini-3.1-pro-preview",
        "reasoning_control": "generation_config.thinking_level='low'",
        "answer_quality": 0.90,
        "retrieval_quality": 0.88,
        "base_latency_s": 1.9,
        "input_usd_per_mtok": 2.00,
        "output_usd_per_mtok": 12.00,
        "color": "#4285f4",
    },
    {
        "provider": "Z.ai",
        "model": "glm-5.2",
        "reasoning_control": "thinking disabled for eval, or reasoning_effort='high'/'max'",
        "answer_quality": 0.86,
        "retrieval_quality": 0.84,
        "base_latency_s": 1.7,
        "input_usd_per_mtok": 0.60,
        "output_usd_per_mtok": 2.00,
        "color": "#7c3aed",
    },
    {
        "provider": "DeepSeek",
        "model": "deepseek-v4-pro",
        "reasoning_control": "thinking disabled for eval, or reasoning_effort='high'/'max'",
        "answer_quality": 0.88,
        "retrieval_quality": 0.86,
        "base_latency_s": 1.5,
        "input_usd_per_mtok": 0.435,
        "output_usd_per_mtok": 0.87,
        "color": "#0ea5e9",
    },
]

pd.DataFrame(MODEL_PROFILES)[["provider", "model", "reasoning_control"]]


In [ ]:
def degraded_query(task: dict[str, Any]) -> str:
    return "agent workflow tool evaluation"


def mock_answer(profile: dict[str, Any], task: dict[str, Any], hits: list[dict[str, Any]], retrieval_ok: bool) -> str:
    answer_ok = retrieval_ok and stable_random(profile["model"], task["id"], "answer") <= profile["answer_quality"]
    if answer_ok:
        source = hits[0]["doc_id"] if hits else task["expected_sources"][0]
        return f"{task['expected_answer']} Source: {source}."
    if retrieval_ok:
        source = hits[0]["doc_id"] if hits else task["expected_sources"][0]
        return f"The notes partially support this, but the answer is incomplete. Source: {source}."
    return "I could not find enough evidence in the retrieved notes to answer confidently."


def run_mock_case(profile: dict[str, Any], task: dict[str, Any], k: int = RETRIEVAL_K) -> dict[str, Any]:
    retrieval_ok = stable_random(profile["model"], task["id"], "retrieval") <= profile["retrieval_quality"]
    query = task["question"] if retrieval_ok else degraded_query(task)
    hits = search_notes(query, k=k)

    if not retrieval_ok:
        expected = set(task["expected_sources"])
        alternate_hits = [hit for hit in search_notes(query, k=k + 4) if hit["doc_id"] not in expected]
        hits = alternate_hits[:k] if alternate_hits else hits

    answer = mock_answer(profile, task, hits, retrieval_ok)
    retrieved_sources = [hit["doc_id"] for hit in hits]
    input_tokens = estimate_tokens(task["question"] + tool_payload(hits)) + 90
    output_tokens = estimate_tokens(answer) + 35
    reasoning_tokens = int(output_tokens * (0.25 + stable_random(profile["model"], task["id"], "reasoning") * 0.35))
    total_output_tokens = output_tokens + reasoning_tokens
    cost_usd = (
        input_tokens / 1_000_000 * profile["input_usd_per_mtok"]
        + total_output_tokens / 1_000_000 * profile["output_usd_per_mtok"]
    )
    jitter = stable_random(profile["model"], task["id"], "latency") * 0.7
    latency_s = profile["base_latency_s"] + 0.05 * len(hits) + jitter

    ans_score = keyword_score(answer, task["expected_keywords"])
    hit_score = retrieved_source_hit(retrieved_sources, task["expected_sources"])
    cite_score = citation_score(answer, retrieved_sources)
    return {
        "mode": "mock",
        "provider": profile["provider"],
        "model": profile["model"],
        "task_id": task["id"],
        "question": task["question"],
        "tool_query": query,
        "answer": answer,
        "expected_answer": task["expected_answer"],
        "expected_sources": task["expected_sources"],
        "retrieved_sources": retrieved_sources,
        "retrieval_hit": hit_score,
        "answer_score": ans_score,
        "citation_score": cite_score,
        "faithfulness_score": ans_score * max(0.5, hit_score),
        "latency_s": round(latency_s, 3),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": input_tokens + total_output_tokens,
        "estimated_cost_usd": round(cost_usd, 6),
        "tool_calls": 1,
    }


def run_mock_eval() -> pd.DataFrame:
    rows = []
    for profile in MODEL_PROFILES:
        for task in TASKS:
            rows.append(run_mock_case(profile, task))
    return pd.DataFrame(rows)

results_df = run_mock_eval()
results_df.head(8)[["provider", "model", "task_id", "retrieval_hit", "answer_score", "latency_s", "estimated_cost_usd"]]


## Summarize results

The decision score below weights answer quality most heavily, but still includes retrieval, citation/faithfulness, latency, and estimated cost. Change the weights to match your application.


In [ ]:
def normalize_lower_is_better(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series([1.0] * len(series), index=series.index)
    return 1 - (series - lo) / (hi - lo)

summary = (
    results_df.groupby(["provider", "model"], as_index=False)
    .agg(
        answer_score=("answer_score", "mean"),
        retrieval_hit=("retrieval_hit", "mean"),
        citation_score=("citation_score", "mean"),
        faithfulness_score=("faithfulness_score", "mean"),
        latency_s=("latency_s", "mean"),
        estimated_cost_usd=("estimated_cost_usd", "sum"),
        total_tokens=("total_tokens", "sum"),
    )
)
summary["cost_efficiency"] = normalize_lower_is_better(summary["estimated_cost_usd"])
summary["latency_efficiency"] = normalize_lower_is_better(summary["latency_s"])
summary["decision_score"] = (
    0.45 * summary["answer_score"]
    + 0.20 * summary["retrieval_hit"]
    + 0.15 * summary["faithfulness_score"]
    + 0.10 * summary["cost_efficiency"]
    + 0.10 * summary["latency_efficiency"]
)
summary = summary.sort_values("decision_score", ascending=False).reset_index(drop=True)
summary


In [ ]:
best = summary.iloc[0]
print(
    f"Mock winner: {best['provider']} {best['model']} "
    f"with decision_score={best['decision_score']:.2f}."
)
print(
    "Interpret this as a notebook smoke test, not a benchmark. "
    "Run live calls on your own docs before making a real model choice."
)


## Visualize tradeoffs


In [ ]:
color_by_model = {profile["model"]: profile["color"] for profile in MODEL_PROFILES}
summary_plot = summary.copy()
summary_plot["label"] = summary_plot["provider"] + "\n" + summary_plot["model"]
colors = [color_by_model.get(model, "#64748b") for model in summary_plot["model"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].bar(summary_plot["label"], summary_plot["decision_score"], color=colors)
axes[0].set_title("Decision score")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", rotation=35, labelsize=8)
axes[0].set_ylabel("higher is better")

axes[1].scatter(
    summary_plot["estimated_cost_usd"],
    summary_plot["answer_score"],
    s=(summary_plot["latency_s"] * 70),
    c=colors,
    alpha=0.85,
    edgecolor="white",
    linewidth=0.8,
)
for _, row in summary_plot.iterrows():
    axes[1].annotate(row["model"], (row["estimated_cost_usd"], row["answer_score"]), fontsize=8, xytext=(4, 3), textcoords="offset points")
axes[1].set_title("Quality vs. mock cost")
axes[1].set_xlabel("estimated cost for 10 cases (USD)")
axes[1].set_ylabel("answer score")
axes[1].grid(alpha=0.2)

axes[2].barh(summary_plot["label"], summary_plot["latency_s"], color=colors)
axes[2].invert_yaxis()
axes[2].set_title("Average latency")
axes[2].set_xlabel("seconds")
axes[2].tick_params(axis="y", labelsize=8)

fig.tight_layout()
plt.show()


In [ ]:
metric_cols = ["answer_score", "retrieval_hit", "faithfulness_score", "cost_efficiency", "latency_efficiency"]
heatmap_data = summary.set_index("model")[metric_cols]

fig, ax = plt.subplots(figsize=(9, 4.3))
im = ax.imshow(heatmap_data.values, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(metric_cols)), labels=[c.replace("_", "\n") for c in metric_cols])
ax.set_yticks(range(len(heatmap_data.index)), labels=heatmap_data.index)
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        ax.text(j, i, f"{heatmap_data.iloc[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8)
ax.set_title("Model comparison heatmap")
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
fig.tight_layout()
plt.show()


In [ ]:
per_task = results_df.pivot_table(index="task_id", columns="model", values="answer_score", aggfunc="mean")
per_task = per_task[[profile["model"] for profile in MODEL_PROFILES]]
per_task.style.format("{:.2f}").background_gradient(cmap="YlGn", axis=None, vmin=0, vmax=1)


## Optional live API runners

The next cell defines live runners for the providers. It does not call any APIs by itself.

Default cost controls:

- `RUN_LIVE_API_CALLS = False`
- `LIVE_CASE_LIMIT = 2`
- `RETRIEVAL_K = 3`
- `MAX_OUTPUT_TOKENS = 650`
- reasoning/thinking set low or disabled for the first pass

Recommended workflow for class:

1. Run the mock notebook end to end.
2. Turn on one provider at a time.
3. Run 2 cases first.
4. Inspect traces and costs.
5. Increase to all 10 cases only after the first smoke test looks right.


In [ ]:
SEARCH_TOOL_PARAMETERS = {
    "type": "object",
    "properties": {
        "query": {"type": "string", "description": "Search query for the local notes corpus."},
        "k": {"type": "integer", "minimum": 1, "maximum": 5, "description": "Number of chunks to return."},
    },
    "required": ["query"],
    "additionalProperties": False,
}

OPENAI_RESPONSES_TOOL = {
    "type": "function",
    "name": "search_notes",
    "description": "Search the local course-notes folder and return relevant chunks with doc_id and chunk_id.",
    "parameters": SEARCH_TOOL_PARAMETERS,
    "strict": True,
}

CHAT_COMPLETIONS_TOOL = {
    "type": "function",
    "function": {
        "name": "search_notes",
        "description": "Search the local course-notes folder and return relevant chunks with doc_id and chunk_id.",
        "parameters": SEARCH_TOOL_PARAMETERS,
    },
}

ANTHROPIC_TOOL = {
    "name": "search_notes",
    "description": (
        "Search only the local course-notes folder. Use this before answering questions "
        "that depend on the notes. Return compact chunks with source identifiers."
    ),
    "input_schema": SEARCH_TOOL_PARAMETERS,
}

GEMINI_TOOL = {
    "type": "function",
    "name": "search_notes",
    "description": "Search the local course-notes folder and return relevant chunks with doc_id and chunk_id.",
    "parameters": SEARCH_TOOL_PARAMETERS,
}


def object_to_dict(obj: Any) -> dict[str, Any]:
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump(exclude_none=True)
    if hasattr(obj, "dict"):
        return obj.dict()
    out = {}
    for name in ("input_tokens", "output_tokens", "total_tokens", "prompt_tokens", "completion_tokens"):
        if hasattr(obj, name):
            out[name] = getattr(obj, name)
    return out


def score_live_record(record: dict[str, Any], task: dict[str, Any]) -> dict[str, Any]:
    retrieved_sources = record.get("retrieved_sources", [])
    answer = record.get("answer", "") or ""
    ans_score = keyword_score(answer, task["expected_keywords"])
    hit_score = retrieved_source_hit(retrieved_sources, task["expected_sources"])
    cite = citation_score(answer, retrieved_sources)
    record.update({
        "expected_answer": task["expected_answer"],
        "expected_sources": task["expected_sources"],
        "retrieval_hit": hit_score,
        "answer_score": ans_score,
        "citation_score": cite,
        "faithfulness_score": ans_score * max(0.5, hit_score),
    })
    return record


def run_openai_case(task: dict[str, Any], model: str = "gpt-5.5") -> dict[str, Any]:
    from openai import OpenAI

    client = OpenAI()
    input_items: list[Any] = [{"role": "user", "content": task["question"]}]
    api_calls = []
    retrieved_sources: list[str] = []
    tool_queries: list[str] = []
    final_answer = ""

    for _ in range(4):
        t0 = time.perf_counter()
        response = client.responses.create(
            model=model,
            instructions=(
                "Answer using the search_notes tool when the question depends on the course notes. "
                "Cite doc_id or chunk_id values. If evidence is missing, say NOT_FOUND."
            ),
            input=input_items,
            tools=[OPENAI_RESPONSES_TOOL],
            reasoning={"effort": "low"},
            max_output_tokens=MAX_OUTPUT_TOKENS,
            parallel_tool_calls=False,
            store=False,
        )
        api_calls.append({"latency_s": time.perf_counter() - t0, "usage": object_to_dict(response.usage)})
        input_items.extend([item.model_dump(exclude_none=True) if hasattr(item, "model_dump") else item for item in response.output])
        calls = [item for item in response.output if getattr(item, "type", None) == "function_call"]
        if not calls:
            final_answer = getattr(response, "output_text", "") or ""
            break
        for call in calls:
            args = json.loads(call.arguments or "{}")
            query = args.get("query", task["question"])
            k = int(args.get("k", RETRIEVAL_K))
            hits = search_notes(query, k=k)
            tool_queries.append(query)
            retrieved_sources.extend([hit["doc_id"] for hit in hits])
            input_items.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": tool_payload(hits),
            })

    record = {
        "mode": "live",
        "provider": "OpenAI",
        "model": model,
        "task_id": task["id"],
        "question": task["question"],
        "tool_query": " | ".join(tool_queries),
        "answer": final_answer,
        "retrieved_sources": sorted(set(retrieved_sources)),
        "latency_s": round(sum(call["latency_s"] for call in api_calls), 3),
        "tool_calls": len(tool_queries),
        "api_calls": api_calls,
    }
    return score_live_record(record, task)


def run_anthropic_case(task: dict[str, Any], model: str = "claude-opus-4-8") -> dict[str, Any]:
    import anthropic

    client = anthropic.Anthropic()
    messages = [{"role": "user", "content": task["question"]}]
    system = (
        "Answer using the search_notes tool when the question depends on course notes. "
        "Cite doc_id or chunk_id values. If evidence is missing, say NOT_FOUND."
    )
    api_calls = []
    retrieved_sources: list[str] = []
    tool_queries: list[str] = []
    final_answer = ""

    for _ in range(4):
        t0 = time.perf_counter()
        msg = client.messages.create(
            model=model,
            max_tokens=MAX_OUTPUT_TOKENS,
            system=system,
            messages=messages,
            tools=[ANTHROPIC_TOOL],
            tool_choice={"type": "auto"},
            temperature=0,
        )
        api_calls.append({"latency_s": time.perf_counter() - t0, "usage": object_to_dict(msg.usage), "stop_reason": msg.stop_reason})
        messages.append({"role": "assistant", "content": msg.content})
        tool_blocks = [block for block in msg.content if getattr(block, "type", None) == "tool_use"]
        if not tool_blocks:
            final_answer = "".join(getattr(block, "text", "") for block in msg.content if getattr(block, "type", None) == "text")
            break
        results = []
        for block in tool_blocks:
            query = block.input.get("query", task["question"])
            k = int(block.input.get("k", RETRIEVAL_K))
            hits = search_notes(query, k=k)
            tool_queries.append(query)
            retrieved_sources.extend([hit["doc_id"] for hit in hits])
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": tool_payload(hits)})
        messages.append({"role": "user", "content": results})

    record = {
        "mode": "live",
        "provider": "Anthropic",
        "model": model,
        "task_id": task["id"],
        "question": task["question"],
        "tool_query": " | ".join(tool_queries),
        "answer": final_answer,
        "retrieved_sources": sorted(set(retrieved_sources)),
        "latency_s": round(sum(call["latency_s"] for call in api_calls), 3),
        "tool_calls": len(tool_queries),
        "api_calls": api_calls,
    }
    return score_live_record(record, task)


def run_openai_compatible_chat_case(
    task: dict[str, Any],
    provider: str,
    model: str,
    api_key_env: str,
    base_url: str,
    extra_body: dict[str, Any] | None = None,
) -> dict[str, Any]:
    from openai import OpenAI

    api_key = os.getenv(api_key_env)
    if not api_key and api_key_env == "ZAI_API_KEY":
        api_key = os.getenv("ZHIPUAI_API_KEY")
    if not api_key:
        raise RuntimeError(f"Missing {api_key_env}")

    client = OpenAI(api_key=api_key, base_url=base_url)
    messages: list[dict[str, Any]] = [
        {"role": "system", "content": "Use search_notes before answering. Cite doc_id or chunk_id values. If evidence is missing, say NOT_FOUND."},
        {"role": "user", "content": task["question"]},
    ]
    api_calls = []
    retrieved_sources: list[str] = []
    tool_queries: list[str] = []
    final_answer = ""

    for turn in range(4):
        kwargs = {
            "model": model,
            "messages": messages,
            "tools": [CHAT_COMPLETIONS_TOOL],
            "tool_choice": "auto",
            "temperature": 0,
            "max_tokens": MAX_OUTPUT_TOKENS,
        }
        if extra_body:
            kwargs["extra_body"] = extra_body
        t0 = time.perf_counter()
        response = client.chat.completions.create(**kwargs)
        api_calls.append({"latency_s": time.perf_counter() - t0, "usage": object_to_dict(response.usage)})
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True) if hasattr(msg, "model_dump") else object_to_dict(msg))
        tool_calls = getattr(msg, "tool_calls", None) or []
        if not tool_calls:
            final_answer = getattr(msg, "content", "") or ""
            break
        for call in tool_calls:
            args = json.loads(call.function.arguments or "{}")
            query = args.get("query", task["question"])
            k = int(args.get("k", RETRIEVAL_K))
            hits = search_notes(query, k=k)
            tool_queries.append(query)
            retrieved_sources.extend([hit["doc_id"] for hit in hits])
            messages.append({"role": "tool", "tool_call_id": call.id, "content": tool_payload(hits)})

    record = {
        "mode": "live",
        "provider": provider,
        "model": model,
        "task_id": task["id"],
        "question": task["question"],
        "tool_query": " | ".join(tool_queries),
        "answer": final_answer,
        "retrieved_sources": sorted(set(retrieved_sources)),
        "latency_s": round(sum(call["latency_s"] for call in api_calls), 3),
        "tool_calls": len(tool_queries),
        "api_calls": api_calls,
    }
    return score_live_record(record, task)


def run_gemini_case(task: dict[str, Any], model: str = "gemini-3.1-pro-preview") -> dict[str, Any]:
    from google import genai

    client = genai.Client()
    api_calls = []
    retrieved_sources: list[str] = []
    tool_queries: list[str] = []

    t0 = time.perf_counter()
    first = client.interactions.create(
        model=model,
        input=task["question"],
        tools=[GEMINI_TOOL],
        generation_config={"thinking_level": "low", "max_output_tokens": MAX_OUTPUT_TOKENS},
    )
    api_calls.append({"latency_s": time.perf_counter() - t0, "usage": object_to_dict(getattr(first, "usage", None))})

    function_steps = [step for step in getattr(first, "steps", []) if getattr(step, "type", None) == "function_call"]
    if function_steps:
        fc = function_steps[0]
        args = getattr(fc, "arguments", {}) or {}
        query = args.get("query", task["question"])
        k = int(args.get("k", RETRIEVAL_K))
        hits = search_notes(query, k=k)
        tool_queries.append(query)
        retrieved_sources.extend([hit["doc_id"] for hit in hits])
        t1 = time.perf_counter()
        final = client.interactions.create(
            model=model,
            previous_interaction_id=first.id,
            tools=[GEMINI_TOOL],
            input=[{
                "type": "function_result",
                "name": fc.name,
                "call_id": fc.id,
                "result": [{"type": "text", "text": tool_payload(hits)}],
            }],
        )
        api_calls.append({"latency_s": time.perf_counter() - t1, "usage": object_to_dict(getattr(final, "usage", None))})
        final_answer = getattr(final, "output_text", "") or ""
    else:
        final_answer = getattr(first, "output_text", "") or ""

    record = {
        "mode": "live",
        "provider": "Google",
        "model": model,
        "task_id": task["id"],
        "question": task["question"],
        "tool_query": " | ".join(tool_queries),
        "answer": final_answer,
        "retrieved_sources": sorted(set(retrieved_sources)),
        "latency_s": round(sum(call["latency_s"] for call in api_calls), 3),
        "tool_calls": len(tool_queries),
        "api_calls": api_calls,
    }
    return score_live_record(record, task)


LIVE_RUNNERS = {
    "gpt-5.5": lambda task: run_openai_case(task, model="gpt-5.5"),
    "claude-opus-4-8": lambda task: run_anthropic_case(task, model="claude-opus-4-8"),
    "gemini-3.1-pro-preview": lambda task: run_gemini_case(task, model="gemini-3.1-pro-preview"),
    "deepseek-v4-pro": lambda task: run_openai_compatible_chat_case(
        task,
        provider="DeepSeek",
        model="deepseek-v4-pro",
        api_key_env="DEEPSEEK_API_KEY",
        base_url="https://api.deepseek.com",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "glm-5.2": lambda task: run_openai_compatible_chat_case(
        task,
        provider="Z.ai",
        model="glm-5.2",
        api_key_env="ZAI_API_KEY",
        base_url="https://open.bigmodel.cn/api/paas/v4/",
        extra_body={"thinking": {"type": "disabled"}},
    ),
}


In [ ]:
# Live run switch. Leave this disabled until you have checked API keys and pricing.
LIVE_MODELS_TO_RUN = ["gpt-5.5"]

if RUN_LIVE_API_CALLS:
    live_rows = []
    for model in LIVE_MODELS_TO_RUN:
        runner = LIVE_RUNNERS[model]
        for task in TASKS[:LIVE_CASE_LIMIT]:
            print(f"Running {model} on {task['id']}...")
            live_rows.append(runner(task))
    live_results_df = pd.DataFrame(live_rows)
    display(live_results_df[["provider", "model", "task_id", "retrieval_hit", "answer_score", "latency_s", "tool_calls"]])
else:
    print("RUN_LIVE_API_CALLS is False. Using mock results only.")


## How to use the result

For this document-QA task, the real decision should come from live runs on your own notes and questions, not from public benchmarks alone. Public leaderboards help pick candidates, but this notebook measures the exact behavior you care about:

- Does the model call retrieval when it should?
- Does the retrieval query find the expected note?
- Does the answer use retrieved evidence instead of memory?
- How much latency and cost does the tool loop add?
- Does extra reasoning effort improve the answer enough to justify the price?

A good next experiment is to run `gpt-5.5` at `reasoning.effort='low'`, `'medium'`, and `'high'` on the same 10 cases. That usually teaches more than comparing five providers once.
